# All-in-One Image Restoration — Pilot Experiment v2 (Fair Pilot Execution)
**Quy trình thực thi sòng phẳng**:
1. Kiểm tra hình học Prompt Cosine (`measure_prompt_cosine.py`)
2. Smoke test thật (`train_pilot_v2.py --smoke_test`, ghi vào `runs/smoke/`, không đụng `results/`)
3. Đo mức nền Iteration 0 (`eval_iter0.py` trên Nhánh A và D, xác nhận B/C trùng A)
4. Chẩn đoán thất bại miền tần số FFT (`diagnose_failures.py`)
5. Huấn luyện Nhánh A x 3 seeds (42, 123, 2024)
6. Khóa ngưỡng thống kê dựa trên $\hat{\sigma}_A$ (`thresholds_locked.json`)
7. Huấn luyện Nhánh B (Guard check `thresholds_locked.json`)
8. Huấn luyện Nhánh C (Guard check `thresholds_locked.json`)
9. Tổng hợp kết quả & Ma trận quyết định khoa học (Gate P4)

## 1. Thiết Lập Môi Trường & Kiểm Tra Phần Cứng GPU Tesla T4

In [ ]:
# Kiểm tra GPU phần cứng
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Sẵn sàng: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Thiết bị GPU: {torch.cuda.get_device_name(0)}")
    print(f"Dung lượng VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")


## 2. Đồng Bộ Codebase Sang Đĩa SSD Cục Bộ Colab

In [ ]:
import os, shutil
from pathlib import Path

%cd /content
# Dọn sạch để đảm bảo cập nhật mã nguồn mới nhất và tránh cấu trúc lồng nhau
!rm -rf /content/repo /content/temp_repo

print("Đang clone codebase từ GitHub...")
!git clone https://github.com/nthmaiii/all-in-one-image-restoration.git /content/temp_repo

# Đưa nội dung thư mục repo/ ra /content/repo và giữ thư mục .git để phục vụ audit commit HEAD
shutil.move("/content/temp_repo/repo", "/content/repo")
shutil.move("/content/temp_repo/.git", "/content/repo/.git")
!rm -rf /content/temp_repo

%cd /content/repo
!pip install -q -r requirements.txt

# In và kiểm toán Git Commit HEAD
commit_hash = !git rev-parse HEAD
print(f">> [AUDIT KHOA HỌC] Git Commit HEAD: {commit_hash[0]}")
os.makedirs("runs/pilot_v2", exist_ok=True)
with open("runs/pilot_v2/git_commit.txt", "w", encoding="utf-8") as f:
    f.write(commit_hash[0] + "\n")
print(">> Môi trường thực thi Pilot v2 đã sẵn sàng!")


## 3. Chuẩn Bị Dữ Liệu v2 & Kiểm Tra Chống Rò Rỉ (Zero-Leakage Assertion)

In [ ]:
# 1. Tải trọng số PromptIR và tập test sạch (BSD68, Kodak24) cho Held-out Primary Endpoint
!python scripts/download_datasets.py --pilot_v2

# 2. Chuẩn bị 160 ảnh train sạch và 40 ảnh val sạch độc lập hoàn toàn với test & kiểm tra zero-leakage
!python scripts/prepare_data_v2.py


## 4. Kiểm Tra Hình Học Prompt Cosine (Cosine Check)

In [ ]:
# Đo đạc ma trận Gram Cosine thực tế giữa các prompt tokens của PromptIR pretrained
!python scripts/measure_prompt_cosine.py --checkpoint baselines/weights/promptir_all.ckpt


## 5. Smoke Test Thật (Chạy Evaluator Thật trên Tập Con Nhỏ, Ghi vào runs/smoke/)

In [ ]:
# Chạy smoke test thật: duyệt toàn bộ pipeline (train + eval 4 tác vụ + hybrid ood), ghi vào runs/smoke/
!python scripts/train_pilot_v2.py --branch A --smoke_test --seed 42 --patch_size 128
!python scripts/train_pilot_v2.py --branch B --smoke_test --seed 42 --patch_size 128
!python scripts/train_pilot_v2.py --branch C --smoke_test --seed 42 --patch_size 128
print(">> Smoke test hoàn tất thành công! Kết quả lưu riêng tại runs/smoke/, tuyệt đối không đụng vào results/.")


## 6. Đo Mức Nền Tại Iteration 0 (eval_iter0: Nhánh A & D, Xác Nhận B/C Zero-Init)

In [ ]:
# Đo PSNR mức nền tại Iteration 0 trước fine-tuning cho PromptIR pretrained và xác nhận B, C trùng A
!python scripts/eval_iter0.py --weights baselines/weights/promptir_all.ckpt


## 7. Chẩn Đoán Thất Bại & Phân Tích Phổ Tần Số FFT (FFT Diagnosis)

In [ ]:
# Phân tích phổ tần số và lỗi tương đối chuẩn hóa diện tích (RBE) phục vụ chẩn đoán cơ chế
!python scripts/diagnose_failures.py --weights baselines/weights/promptir_all.ckpt


## 8. Nhánh A: Fine-tune PromptIR Baseline (3 Seeds: 42, 123, 2024)

In [ ]:
# Nhánh A: PromptIR Baseline (chạy trước để xác định độ lệch chuẩn nền và khóa ngưỡng)
!python scripts/train_pilot_v2.py --branch A --seed 42 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch A --seed 123 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch A --seed 2024 --iters 1500 --batch_size 4 --patch_size 128


## 9. Khóa Ngưỡng Thống Kê Dựa Trên σ_A (thresholds_locked.json)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("results/pilot_v2_results.csv")
assert csv_path.exists(), "Không tìm thấy file kết quả results/pilot_v2_results.csv từ Nhánh A!"

df = pd.read_csv(csv_path)
df_a_final = df[(df["branch"] == "A") & (df["checkpoint_type"] == "final")]
assert len(df_a_final) == 3, f"Nhánh A phải hoàn tất đủ 3 seeds trước khi khóa ngưỡng! Hiện có: {len(df_a_final)}"

psnr_a = df_a_final["primary_endpoint_ood_psnr"].values
mu_a = float(np.mean(psnr_a))
sigma_a = float(np.std(psnr_a, ddof=1))

# Công thức khóa ngưỡng duy nhất theo quy chuẩn kiểm toán:
X = float(max(0.40, 2.0 * sigma_a))
Z = float(max(0.25, 1.5 * sigma_a))

thresholds = {
    "mu_A": round(mu_a, 4),
    "sigma_A": round(sigma_a, 4),
    "X_threshold_B_vs_A": round(X, 4),
    "Z_threshold_B_vs_C": round(Z, 4),
    "single_task_tol_mean": -0.15,
    "single_task_tol_seed": -0.30,
    "seeds_A": df_a_final["seed"].tolist(),
    "psnr_A_per_seed": [round(x, 4) for x in psnr_a.tolist()]
}

with open("thresholds_locked.json", "w", encoding="utf-8") as f:
    json.dump(thresholds, f, indent=2)

print("=" * 60)
print("ĐÃ KHÓA NGƯỠNG ĐÁNH GIÁ THỐNG KÊ (THRESHOLDS LOCKED):")
print(f"-> Mean PSNR A (Primary Endpoint): {mu_a:.4f} dB")
print(f"-> Std PSNR A (sigma_A):            {sigma_a:.4f} dB")
print(f"-> Ngưỡng X (B - A >= X):           {X:.4f} dB")
print(f"-> Ngưỡng Z (B - C >= Z):           {Z:.4f} dB")
print("Đã lưu thành công vào file: thresholds_locked.json")
print("=" * 60)


## 10. Nhánh B: Huấn Luyện FD-PromptIR Đề Xuất (3 Seeds: 42, 123, 2024)

In [ ]:
from pathlib import Path
assert Path("thresholds_locked.json").exists(), (
    "[GATE GUARD ABORT] Không tìm thấy file thresholds_locked.json! "
    "Quy chuẩn thực nghiệm bắt buộc phải chạy hoàn tất Nhánh A x 3 seeds và khóa ngưỡng trước khi chạy Nhánh B!"
)

# Nhánh B: FD-PromptIR (3 seeds: 42, 123, 2024)
!python scripts/train_pilot_v2.py --branch B --seed 42 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch B --seed 123 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch B --seed 2024 --iters 1500 --batch_size 4 --patch_size 128


## 11. Nhánh C: Đối Chứng Tham Số Param-Matched PromptIR (3 Seeds: 42, 123, 2024)

In [ ]:
from pathlib import Path
assert Path("thresholds_locked.json").exists(), (
    "[GATE GUARD ABORT] Không tìm thấy file thresholds_locked.json! "
    "Quy chuẩn thực nghiệm bắt buộc phải chạy hoàn tất Nhánh A x 3 seeds và khóa ngưỡng trước khi chạy Nhánh C!"
)

# Nhánh C: Param-Matched PromptIR (3 seeds: 42, 123, 2024)
!python scripts/train_pilot_v2.py --branch C --seed 42 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch C --seed 123 --iters 1500 --batch_size 4 --patch_size 128
!python scripts/train_pilot_v2.py --branch C --seed 2024 --iters 1500 --batch_size 4 --patch_size 128


## 12. Tổng Hợp Kết Quả & Kiểm Định Tiêu Chí Thắng/Thua Chuẩn Hóa (Gate P4)

In [ ]:
import json
import pandas as pd
import numpy as np
from pathlib import Path

csv_path = Path("results/pilot_v2_results.csv")
thresh_path = Path("thresholds_locked.json")

if not csv_path.exists() or not thresh_path.exists():
    print("Chưa đủ dữ liệu kết quả hoặc file thresholds_locked.json!")
else:
    with open(thresh_path, "r", encoding="utf-8") as f:
        thresh = json.load(f)
    
    df = pd.read_csv(csv_path)
    df_final = df[df["checkpoint_type"] == "final"].copy()
    
    print("=" * 70)
    print("BẢNG TỔNG HỢP FINAL MODEL TRÊN PRIMARY ENDPOINT (HELD-OUT HYBRID)")
    print("=" * 70)
    cols = ["branch", "seed", "val_psnr_noise", "val_psnr_rain", "val_psnr_haze", "val_psnr_hybrid", "primary_endpoint_ood_psnr", "gpu_hours"]
    print(df_final[cols].to_markdown(index=False))
    
    # Tính mean và seed diffs
    mu = df_final.groupby("branch")["primary_endpoint_ood_psnr"].mean().to_dict()
    mu_B = mu.get("B", 0.0)
    mu_A = thresh["mu_A"]
    mu_C = mu.get("C", 0.0)
    delta_BA = mu_B - mu_A
    delta_BC = mu_B - mu_C
    X = thresh["X_threshold_B_vs_A"]
    Z = thresh["Z_threshold_B_vs_C"]
    
    p_B = df_final[df_final["branch"] == "B"].sort_values("seed")["primary_endpoint_ood_psnr"].values
    p_A = df_final[df_final["branch"] == "A"].sort_values("seed")["primary_endpoint_ood_psnr"].values
    p_C = df_final[df_final["branch"] == "C"].sort_values("seed")["primary_endpoint_ood_psnr"].values
    
    all_seeds_BA_pos = bool(np.all(p_B - p_A > 0)) if len(p_B) == 3 and len(p_A) == 3 else False
    all_seeds_BC_pos = bool(np.all(p_B - p_C > 0)) if len(p_B) == 3 and len(p_C) == 3 else False
    
    print("\n" + "=" * 70)
    print(f">> Delta(B - A) = {delta_BA:+.4f} dB (Ngưỡng X = {X:.4f} dB) -> Đạt: {delta_BA >= X}")
    print(f">> Delta(B - C) = {delta_BC:+.4f} dB (Ngưỡng Z = {Z:.4f} dB) -> Đạt: {delta_BC >= Z}")
    print(f">> Ràng buộc từng Seed: B > A trên 3/3 seeds: {all_seeds_BA_pos} | B > C trên 3/3 seeds: {all_seeds_BC_pos}")
    print("=" * 70)
